# COVID-CT classification with Fed-BioMed

Federated binary classification (COVID vs NonCOVID) of CT slices from the UCSD-AI4H COVID-CT dataset,
over two hospitals with a non-IID split. See `README_COVIDCT.md` for data preparation and node setup.

Before running this notebook:

1. `python download_and_split_covidct.py -f <download-folder>` created `hospital-a` and `hospital-b`,
2. each node registered its `data/train` folder as an **images** dataset tagged `covid-ct`,
3. the nodes are running (`fedbiomed node -p hospital-a start`, `fedbiomed node -p hospital-b start`).

## Training plan

`ImageFolderDataset` reads `<root>/<class>/<image>` folders: class labels come from the sorted folder names
(`CT_COVID` = 0, `CT_NonCOVID` = 1). Images are loaded as RGB and handed to the transform as tensors in `[0, 1]`,
so only tensor-compatible transforms are used.

In [ ]:
import torch.nn as nn
from torch.optim import Adam
from torchvision import models, transforms

from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.dataset import ImageFolderDataset
from fedbiomed.common.training_plans import TorchTrainingPlan


class CovidCTClassificationPlan(TorchTrainingPlan):
    def init_model(self, model_args):
        # No pretrained weights: nodes must not download anything
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, model_args["num_classes"])
        return model

    def init_optimizer(self, optimizer_args):
        return Adam(self.model().parameters(), lr=optimizer_args["lr"])

    def init_dependencies(self):
        return [
            "from torch.optim import Adam",
            "from torchvision import models, transforms",
            "from fedbiomed.common.dataset import ImageFolderDataset",
        ]

    def training_data(self):
        transform = transforms.Compose(
            [
                transforms.Resize((224, 224), antialias=True),
                transforms.RandomHorizontalFlip(),
                transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
            ]
        )
        return DataManager(ImageFolderDataset(transform=transform), shuffle=True)

    def training_step(self, data, target):
        output = self.model().forward(data)
        return nn.functional.cross_entropy(output, target)

## Experiment

In [ ]:
from fedbiomed.common.metrics import MetricTypes
from fedbiomed.researcher.aggregators.fedavg import FedAverage
from fedbiomed.researcher.federated_workflows import Experiment

model_args = {"num_classes": 2}

training_args = {
    "loader_args": {"batch_size": 16},
    "optimizer_args": {"lr": 1e-4},
    "epochs": 1,
    "test_ratio": 0.15,
    "test_on_global_updates": True,
    "test_metric": MetricTypes.ACCURACY,
    "random_seed": 21,
}

exp = Experiment(
    tags=["covid-ct"],
    model_args=model_args,
    training_plan_class=CovidCTClassificationPlan,
    training_args=training_args,
    round_limit=10,
    aggregator=FedAverage(),
    node_selection_strategy=None,
)

In [ ]:
exp.run()

## Evaluate on the holdout split

Register `data/holdout` on each node with the tag `covid-ct-holdout` (`dataset add --file <node>/dataset_holdout.json`),
then validate the trained model on it. With `test_ratio=1.0` the whole holdout folder is used for validation
on the global parameters.

**Known limitation:** the node reports the holdout `ACCURACY`, then the round ends with
`FB300 ... 'NoneType' object has no attribute 'dataset'`, because the train split is empty
(`FIXME` in `fedbiomed/node/round.py`). The validation result printed before that error is valid.

In [ ]:
test_exp = Experiment(
    tags=["covid-ct-holdout"],
    model_args=model_args,
    training_plan_class=CovidCTClassificationPlan,
    training_args={**training_args, "test_ratio": 1.0},
    round_limit=1,
    aggregator=FedAverage(),
)
test_exp.training_plan().set_model_params(exp.training_plan().get_model_params())
test_exp.run()

## Optional: capability guardian

For nodes with a `guardian_service` configured, attach a capability minted for the exact training plan source.
See the *Enabling the capability guardian* section of `README.md`.

In [ ]:
import hashlib

from mint_capability import mint_capability

source = exp.training_plan().source()
digest = "sha256:" + hashlib.sha256(source.encode("utf-8")).hexdigest()
exp.set_capabilities(mint_capability(digest))